In [ ]:
!pip install easyocr -q
!pip install ultralytics -q

Used easyOCR here just as a test, since in googleColab there were dependency issues with paddleOCR

In [ ]:
import easyocr
from ultralytics import YOLO
import cv2
import numpy as np
import json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Charger les modèles
yolo   = YOLO('/content/drive/MyDrive/ederest_v6_best.pt')
reader = easyocr.Reader(['fr', 'en'], gpu=True)
print(" YOLO + EasyOCR chargés !")

CLASSES = ['button', 'checkbox', 'dropdown', 'radio button', 'textbox', 'popup']

# Couleur par type d'élément
COLORS = {
    'button'      : '#FF6B6B',
    'checkbox'    : '#4ECDC4',
    'dropdown'    : '#45B7D1',
    'radio button': '#96CEB4',
    'textbox'     : '#FFEAA7',
    'popup'       : '#DDA0DD'
}

# ============================================================
# OCR SUR TOUT L'ÉCRAN
# ============================================================
def ocr_full_screen(image_path):
    result      = reader.readtext(image_path)
    text_blocks = []

    for (bbox, text, confidence) in result:
        if confidence < 0.5:
            continue

        x1 = int(min(p[0] for p in bbox))
        y1 = int(min(p[1] for p in bbox))
        x2 = int(max(p[0] for p in bbox))
        y2 = int(max(p[1] for p in bbox))

        text_blocks.append({
            "text"      : text,
            "bbox"      : [x1, y1, x2, y2],
            "confidence": round(confidence, 3),
            "center"    : [(x1+x2)//2, (y1+y2)//2]
        })

    return text_blocks


def merge_nearby_text_blocks(text_blocks, max_gap=8):
    """
    Fusionne les blocs de texte proches sur la même ligne.
    Résout le problème des labels détectés en plusieurs morceaux.
    """
    if not text_blocks:
        return text_blocks

    # Trier par position Y puis X
    sorted_blocks = sorted(text_blocks, key=lambda b: (b['center'][1], b['center'][0]))
    merged = []
    used   = set()

    for i, block in enumerate(sorted_blocks):
        if i in used:
            continue

        current_text  = block['text']
        current_bbox  = block['bbox'].copy()
        current_y     = block['center'][1]
        used.add(i)

        # Chercher les blocs voisins sur la même ligne
        for j, other in enumerate(sorted_blocks):
            if j in used or j == i:
                continue

            other_y = other['center'][1]

            # Même ligne (±10px)
            if abs(other_y - current_y) > 10:
                continue

            # Gap horizontal acceptable
            gap = other['bbox'][0] - current_bbox[2]  # x1_other - x2_current
            if 0 <= gap <= max_gap:
                # Merger !
                current_text  += " " + other['text']
                current_bbox[2] = other['bbox'][2]  # étendre x2
                current_bbox[3] = max(current_bbox[3], other['bbox'][3])
                used.add(j)

        merged.append({
            "text"      : current_text,
            "bbox"      : current_bbox,
            "confidence": block['confidence'],
            "center"    : [(current_bbox[0]+current_bbox[2])//2,
                           (current_bbox[1]+current_bbox[3])//2]
        })

    return merged

# ============================================================
# YOLO DÉTECTION
# ============================================================
def yolo_detect(image_path):
    results    = yolo.predict(image_path, conf=0.3, verbose=False)
    detections = results[0].boxes
    elements   = []

    for box in detections:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        class_id        = int(box.cls[0])
        confidence      = float(box.conf[0])

        elements.append({
            "type"      : CLASSES[class_id],
            "bbox"      : [x1, y1, x2, y2],
            "confidence": round(confidence, 3),
            "center"    : [(x1+x2)//2, (y1+y2)//2],
            "label"     : "",
            "value"     : ""
        })

    return elements


def filter_elements_in_popup(elements):
    """
    Si un popup est détecté → garder seulement les éléments
    qui sont à l'intérieur de la bbox du popup.
    Ignorer les éléments de l'écran derrière.
    """
    popups = [el for el in elements if el['type'] == 'popup']

    if not popups:
        return elements  # pas de popup → retourner tout

    # Prendre le popup le plus grand
    popup = max(popups, key=lambda p: 
                (p['bbox'][2]-p['bbox'][0]) * (p['bbox'][3]-p['bbox'][1]))

    px1, py1, px2, py2 = popup['bbox']

    # Garder le popup + les éléments dedans
    filtered = [popup]
    for el in elements:
        if el['type'] == 'popup':
            continue
        cx, cy = el['center']
        if px1 <= cx <= px2 and py1 <= cy <= py2:
            filtered.append(el)

    return filtered

# ============================================================
# ENRICHIR SELON LE TYPE
# ============================================================
def find_value(el, text_blocks):
    """Texte à l'INTÉRIEUR de la bbox."""
    x1, y1, x2, y2 = el['bbox']
    inside = []
    for tb in text_blocks:
        tcx, tcy = tb['center']
        if x1 <= tcx <= x2 and y1 <= tcy <= y2:
            inside.append(tb['text'])
    return " ".join(inside).strip()


def find_label(el, text_blocks):
    """
    Cherche uniquement le texte à GAUCHE sur la MÊME ligne.
    Exclut explicitement tout ce qui est en dessous.
    """
    x1, y1, x2, y2 = el['bbox']
    el_height  = y2 - y1
    center_y   = (y1 + y2) / 2
    best_label = ""
    min_dist   = float('inf')

    for tb in text_blocks:
        tx1, ty1, tx2, ty2 = tb['bbox']
        tcx = (tx1 + tx2) / 2
        tcy = (ty1 + ty2) / 2

        is_left     = tcx < x1                      # strictement à gauche
        not_below   = tcy <= center_y + 5           # ← exclut ce qui est en dessous
        same_line   = abs(tcy - center_y) < el_height * 0.6  # même ligne proportionnelle
        not_too_far = (x1 - tcx) < 300              # pas trop loin

        if is_left and not_below and same_line and not_too_far:
            dist = x1 - tcx
            if dist < min_dist:
                min_dist   = dist
                best_label = tb['text']

    return best_label


def find_label_right(el, text_blocks):
    """
    Cherche uniquement le texte à GAUCHE sur la MÊME ligne.
    Exclut explicitement tout ce qui est en dessous.
    """
    x1, y1, x2, y2 = el['bbox']
    el_height  = y2 - y1
    center_y   = (y1 + y2) / 2
    best_label = ""
    min_dist   = float('inf')

    for tb in text_blocks:
        tx1, ty1, tx2, ty2 = tb['bbox']
        tcx = (tx1 + tx2) / 2
        tcy = (ty1 + ty2) / 2

        is_left     = tcx > x1                      # strictement à gauche
        not_below   = tcy <= center_y + 5           # ← exclut ce qui est en dessous
        same_line   = abs(tcy - center_y) < el_height * 0.6  # même ligne proportionnelle
        not_too_far = (x1 - tcx) < 300              # pas trop loin

        if is_left and not_below and same_line and not_too_far:
            dist = x1 - tcx
            if dist < min_dist:
                min_dist   = dist
                best_label = tb['text']

    return best_label


def enrich_elements(yolo_elements, text_blocks_raw, text_blocks_merged):
    """
    - find_value utilise text_blocks_raw    → pas de merge → boutons corrects
    - find_label utilise text_blocks_merged → avec merge   → labels complets
    """
    for el in yolo_elements:
        t = el['type']

        if t == 'button':
            el['value'] = find_value(el, text_blocks_raw)     # ← raw
            el['label'] = ""

        elif t == 'textbox':
            el['label'] = find_label(el, text_blocks_merged)  # ← mergé
            el['value'] = find_value(el, text_blocks_raw)     # ← raw
          
        elif t == 'dropdown':
            el['value'] = find_value(el, text_blocks_raw)     
            el['label'] = find_label(el, text_blocks_merged)  # ← ajoute cette ligne

        elif t in ['checkbox', 'radio button']:
            el['label'] = find_label(el, text_blocks_merged)  # ← mergé
            el['value'] = ""
            if not (el['label']):
                el['label'] = find_label_right(el, text_blocks_merged)


        elif t == 'popup':
            el['value'] = find_value(el, text_blocks_raw)     # ← raw
            el['label'] = ""

    return yolo_elements

# ============================================================
# VISUALISATION
# ============================================================
def visualize(image_path, elements):
    """Affiche le screenshot avec les bounding boxes annotées."""
    img = Image.open(image_path).convert("RGB")
    fig, ax = plt.subplots(1, 1, figsize=(18, 10))
    ax.imshow(img)

    for el in elements:
        x1, y1, x2, y2 = el['bbox']
        w = x2 - x1
        h = y2 - y1

        color = COLORS.get(el['type'], '#FFFFFF')

        # Bounding box
        rect = patches.Rectangle(
            (x1, y1), w, h,
            linewidth=2,
            edgecolor=color,
            facecolor=color,
            alpha=0.25
        )
        ax.add_patch(rect)

        # Label affiché sur la bbox
        display_text = el['value'] if el['value'] else el['label']
        display_text = f"[{el['type']}] {display_text[:20]}"

        ax.text(
            x1, y1 - 5,
            display_text,
            fontsize=7,
            color='white',
            bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none')
        )

    ax.axis('off')
    ax.set_title(image_path.split('/')[-1], fontsize=10)
    plt.tight_layout()
    plt.show()



In [ ]:
import cv2
import numpy as np

def screens_are_similar(screen1_path, screen2_path, threshold=0.97):
    """
    Compare 2 screenshots.
    Retourne True si les layouts sont similaires (même interface).
    """
    img1 = cv2.imread(screen1_path, cv2.IMREAD_GRAYSCALE)
    img2 = cv2.imread(screen2_path, cv2.IMREAD_GRAYSCALE)

    img1 = cv2.resize(img1, (320, 240))
    img2 = cv2.resize(img2, (320, 240))

    diff       = cv2.absdiff(img1, img2)
    similarity = 1 - (np.sum(diff) / (img1.size * 255))

    return similarity > threshold


def analyze_screenshot_smart(image_path, prev_image_path=None, 
                              cached_elements=None, show_viz=False):
    """
    Version optimisée :
    - Si screen similaire au précédent → réutilise les détections YOLO
    - Lance toujours OCR (le texte peut avoir changé)
    """
    print(f"\n🖼️  Analyse : {image_path.split('/')[-1]}")

    # OCR toujours lancé
    text_blocks_raw    = ocr_full_screen(image_path)
    text_blocks_merged = merge_nearby_text_blocks(text_blocks_raw, max_gap=8)

    # YOLO : seulement si layout différent
    if (prev_image_path is not None and
        cached_elements is not None and
        screens_are_similar(prev_image_path, image_path)):

        print(f"  YOLO skippé — layout identique, cache réutilisé")
        elements = cached_elements

    else:
        print(f"  YOLO lancé — nouveau layout détecté")
        elements = yolo_detect(image_path)

    # Enrichir avec le nouvel OCR
    elements = enrich_elements(elements, text_blocks_raw, text_blocks_merged)
    elements = filter_elements_in_popup(elements)

    if show_viz:
        visualize(image_path, elements)

    return elements, text_blocks_raw, text_blocks_merged

In [ ]:
screens = [
    '/content/drive/MyDrive/PAbatchesfolder/images/b2_049_Confirmation_des_opé_Screenshot_11.3.png',    
    '/content/drive/MyDrive/PAbatchesfolder/images/b3_001_Confirmation_des_opé_Screenshot_10.0.png',
    '/content/drive/MyDrive/PAbatchesfolder/images/b1_016_Affichage_Contrat_Af_image_2025-04-22_105757465_662ef0714dd24eb79def51c59f786ea8.png',
]

prev_path       = None
cached_elements = None
all_results     = []

for screen in screens:
    elements, tb_raw, tb_merged = analyze_screenshot_smart(
        screen,
        prev_image_path=prev_path,
        cached_elements=cached_elements
    )

    all_results.append({
        "screenshot": screen,
        "elements"  : elements
    })

    # Mettre à jour le cache
    prev_path       = screen
    cached_elements = yolo_detect(screen)  # cache les détections brutes